# Analiza "połkniętych" bloków w cache Vision

Problem: `json_repair` w `vision_response.clean_response` czasem "połyka" resztę odpowiedzi Vision jako treść jednego bloku — gdy w strumieniu pojawi się nieescape'owany ASCII `"` w kontekście literałów z `\n`. Efekt: kilka kolejnych bloków (w tym section-header) znika jako osobne elementy i trafia jako surowy JSON do `text` bloku poprzedzającego.

Cel notebooka:
1. Inwentaryzacja: ile stron i ile zagubionych bloków.
2. Diagnostyka per strona (co dokładnie przepadło, czy są section-headery z ToC).
3. PoC rekonstrukcji z `text` (split po separatorze + alt. json_repair).
4. Ocena ryzyka i wpływu na pozostałe bloki.
5. Rekomendacja co dalej.

## Sekcja 1 — Inwentaryzacja

In [1]:
from __future__ import annotations
import json
import re
import sys
from pathlib import Path

import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
CACHE = REPO / "data" / "extraction_v2"
CHAPTER_IDS = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X"]
NESTED_MARK = '"element_type"'

rows = []
for cid in CHAPTER_IDS:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    for p in ch["pages"]:
        for b in p["blocks"]:
            t = b.get("text", "") or ""
            c = t.count(NESTED_MARK)
            if c > 0:
                rows.append({
                    "chapter_id": cid,
                    "page_num": p["page_num"],
                    "block_id": b["block_id"],
                    "outer_element_type": b["element_type"],
                    "nested_count": c,
                    "text_length": len(t),
                })
inv = pd.DataFrame(rows).sort_values("nested_count", ascending=False).reset_index(drop=True)
print(f"Stron z połknięciem: {inv['page_num'].nunique()}")
print(f"Łączna liczba zagnieżdżonych element_type: {inv['nested_count'].sum()}")
inv

Stron z połknięciem: 1
Łączna liczba zagnieżdżonych element_type: 6


,chapter_id,page_num,block_id,outer_element_type,nested_count,text_length
0,VII,137,p137_b10,list,6,1555


## Sekcja 2 — Diagnostyka per strona

In [2]:
SECTION_PREFIX_RE = re.compile(r"^\d+\.\s+")

def normalize(s: str) -> str:
    return SECTION_PREFIX_RE.sub("", s, count=1).strip()

def diagnose(chapter_id: str, page_num: int):
    ch = json.loads((CACHE / f"{chapter_id}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == page_num)
    sections_toc = page["sections"]
    sh_on_page = [b for b in page["blocks"] if b["element_type"] == "section-header"]
    sh_texts = [b["text"] for b in sh_on_page]
    missing = [
        s for s in sections_toc
        if normalize(s) not in {normalize(x) for x in sh_texts}
    ]
    swallower = next((b for b in page["blocks"] if NESTED_MARK in (b.get("text") or "")), None)
    print(f"--- {chapter_id} p{page_num} ---")
    print(f"ToC sections: {sections_toc}")
    print(f"Section-header w cache: {sh_texts}")
    print(f"BRAKUJĄCE z ToC: {missing}")
    if swallower:
        nested_in_text = [s for s in sections_toc if normalize(s) in normalize(swallower['text'])]
        print(f"Połykacz: {swallower['block_id']} ({swallower['element_type']}), len={len(swallower['text'])}, nested count={swallower['text'].count(NESTED_MARK)}")
        print(f"ToC-sections znalezione wewnątrz text połykacza: {nested_in_text}")
    print()
    return page, swallower

for cid, pn in [("III", 44), ("VII", 137), ("X", 179)]:
    diagnose(cid, pn)

--- III p44 ---
ToC sections: []
Section-header w cache: []
BRAKUJĄCE z ToC: []

--- VII p137 ---
ToC sections: ['4.  Ryzyko operacyjne', '5.  Ryzyko biznesowe']
Section-header w cache: ['4. Ryzyko operacyjne']
BRAKUJĄCE z ToC: ['5.  Ryzyko biznesowe']
Połykacz: p137_b10 (list), len=1555, nested count=6
ToC-sections znalezione wewnątrz text połykacza: ['5.  Ryzyko biznesowe']

--- X p179 ---
ToC sections: ['8. Metodyka kalkulacji emisji gazów cieplarnianych']
Section-header w cache: ['8. Metodyka kalkulacji emisji gazów cieplarnianych']
BRAKUJĄCE z ToC: []



In [9]:
# Pełny text każdego "połykacza" — do wizualnej inspekcji
for cid, pn in [("VII", 137)]: #[("III", 44), ("VII", 137), ("X", 179)]:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == pn)
    # print(page)
    sw = next((b for b in page["blocks"] if NESTED_MARK in (b.get("text") or "")), None)
    # print(sw)
    print(f"\n========== {cid} p{pn} {sw['block_id']} ==========\n")
    print(sw["text"])


========== VII p137 p137_b10 ==========

- przystosowanie systemów i procedur do wymogów Rozporządzenia CRR3,
- wprowadzenie metody SMA w związku z wejściem w życie Rozporządzenia CRR3,
- dostosowanie wymogów w zakresie bezpieczeństwa cyfrowego zgodnie z Rozporządzeniem „DORA"."},
    {"element_type": "subsection-header", "text": "Charakterystyka ryzyka operacyjnego"},
    {"element_type": "text", "text": "Wartość straty netto w ujęciu księgowym z tytułu zdarzeń ryzyka operacyjnego w 2024 r. wyniosła 22 553 tys. zł. W 2024 r. utrzymujemy akceptowalny profil ryzyka operacyjnego oraz nie przekroczyliśmy przyjętych limitów oraz apetytu na ryzyko operacyjne."},
    {"element_type": "section-header", "text": "5. Ryzyko biznesowe"},
    {"element_type": "subsection-header", "text": "Organizacja systemu zarządzania ryzykiem biznesowym"},
    {"element_type": "text", "text": "Ryzyko biznesowe w BGK rozumiemy jako ryzyko nieosiągnięcia założonych i koniecznych celów ekonomicznych, w szczególno

## Sekcja 3 — Proof-of-concept rekonstrukcji

Dwie strategie dla każdego połykacza:
- **A (split)**: znajdź wszystkie wystąpienia `{"element_type":"...","text":"..."}` w środku `text` regexem tolerancyjnym; wyciągnij z nich parę (type, text). Pozostała część przed pierwszym zagnieżdżonym obiektem to "prawdziwa" treść bloku-połykacza.
- **B (json_repair)**: skleić wrapper `{"elements":[{"element_type":"<outer>","text":"...`+swallower.text+`"}]}` i puścić przez `json_repair`. Czy cokolwiek się zparsuje?

In [10]:
from json_repair import repair_json

# Strategia A: regex — parser zagnieżdżonych {"element_type":"X","text":"Y"}.
# Domknięcie obiektu MUSI być granicą strukturalną kolekcji (nie byle "}).
# Wymagamy: " bezpośrednio przy } (bez whitespace), a po } — przecinek+{"element_type",
# albo ]} (koniec kolekcji), albo koniec inputu. Dzięki temu text może zawierać }, "}, {.
NESTED_OBJ_RE = re.compile(
    r'\{"element_type"\s*:\s*"(?P<et>[^"]+)"\s*,\s*"text"\s*:\s*"(?P<tx>.*?)"\}'
    r'(?=\s*(?:,\s*\{"element_type"|\]\s*\}|$))',
    re.DOTALL,
)

def reconstruct_split(swallower_text: str):
    """Zwraca (prefix_prawdziwej_treści, [(element_type, text), ...])."""
    matches = list(NESTED_OBJ_RE.finditer(swallower_text))
    if not matches:
        return swallower_text, []
    prefix = swallower_text[: matches[0].start()]
    # prefix może kończyć się na '"},' albo '"."},' — obciąć zamknięcie literału
    prefix = re.sub(r'"\.?"\}\s*,?\s*$', "", prefix.rstrip())
    prefix = prefix.rstrip('",. \n\t')
    blocks = [(m.group("et"), m.group("tx")) for m in matches]
    return prefix, blocks

def reconstruct_repair(swallower_text: str, outer_et: str):
    wrapped = (
        '{"elements":[{"element_type":"' + outer_et + '","text":"' + swallower_text + '"}]}'
    )
    try:
        fixed = repair_json(wrapped)
        return json.loads(fixed)
    except Exception as e:
        return {"_error": str(e)}

# PoC na VII p137
ch = json.loads((CACHE / "VII.json").read_text(encoding="utf-8"))
page = next(p for p in ch["pages"] if p["page_num"] == 137)
sw = next(b for b in page["blocks"] if NESTED_MARK in (b.get("text") or ""))

prefix, blocks = reconstruct_split(sw["text"])
print("PREFIX (oryginalna treść bloku-połykacza, po obcięciu):")
print(repr(prefix[:400]))
print(f"\nODZYSKANYCH BLOKÓW: {len(blocks)}")
for et, tx in blocks:
    print(f" - {et}: {tx[:80]!r}")

# Sanity-check: liczba matchy == liczba wystąpień "element_type" w text
# (jeśli jakiś element został ucięty przez zły regex, te liczby się rozjadą)
nested_count = sw["text"].count(NESTED_MARK)
print(f"\nSANITY: matchy={len(blocks)}, nested_count={nested_count}, OK={len(blocks) == nested_count}")

PREFIX (oryginalna treść bloku-połykacza, po obcięciu):
'- przystosowanie systemów i procedur do wymogów Rozporządzenia CRR3,\n- wprowadzenie metody SMA w związku z wejściem w życie Rozporządzenia CRR3,\n- dostosowanie wymogów w zakresie bezpieczeństwa cyfrowego zgodnie z Rozporządzeniem „DORA'

ODZYSKANYCH BLOKÓW: 6
 - subsection-header: 'Charakterystyka ryzyka operacyjnego'
 - text: 'Wartość straty netto w ujęciu księgowym z tytułu zdarzeń ryzyka operacyjnego w 2'
 - section-header: '5. Ryzyko biznesowe'
 - subsection-header: 'Organizacja systemu zarządzania ryzykiem biznesowym'
 - text: 'Ryzyko biznesowe w BGK rozumiemy jako ryzyko nieosiągnięcia założonych i koniecz'
 - text: 'W BGK oceniamy ryzyko biznesowe poprzez identyfikację i analizę wpływu faktyczny'

SANITY: matchy=6, nested_count=6, OK=True


In [11]:
# Tabela porównawcza A vs B dla wszystkich 3 stron
poc_rows = []
for cid, pn in [("III", 44), ("VII", 137), ("X", 179)]:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == pn)
    sw = next(b for b in page["blocks"] if NESTED_MARK in (b.get("text") or ""))
    prefix, blocks_a = reconstruct_split(sw["text"])
    result_b = reconstruct_repair(sw["text"], sw["element_type"])
    b_ok = "_error" not in result_b
    toc = page["sections"]
    sh_in_blocks_a = [et for et, _ in blocks_a if et == "section-header"]
    recovered_sh_texts = [tx for et, tx in blocks_a if et == "section-header"]
    poc_rows.append({
        "chapter_id": cid,
        "page_num": pn,
        "outer_type": sw["element_type"],
        "A_blocks": len(blocks_a),
        "A_section_headers": len(sh_in_blocks_a),
        "A_recovered_sh": recovered_sh_texts,
        "B_parses": b_ok,
        "toc_sections": toc,
    })
pd.DataFrame(poc_rows)

StopIteration: 

## Sekcja 4 — Ocena ryzyka

In [ ]:
# Ryzyko 1: czy naprawa wpływa na INNE bloki na tej samej stronie?
# Sprawdzamy: ile bloków na stronie NIE jest połykaczem — te zostaną nietknięte.
for cid, pn in [("III", 44), ("VII", 137), ("X", 179)]:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == pn)
    total = len(page["blocks"])
    swallowers = sum(1 for b in page["blocks"] if NESTED_MARK in (b.get("text") or ""))
    print(f"{cid} p{pn}: {total} bloków, w tym {swallowers} połykacz(y) — pozostałe {total - swallowers} bez zmian")

# Ryzyko 2: czy prefix (prawdziwa treść bloku-połykacza) odpowiada oryginałowi?
# Porównanie: pierwsze ~N znaków oryginału vs prefix po rekonstrukcji.
print()
for cid, pn in [("III", 44), ("VII", 137), ("X", 179)]:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == pn)
    sw = next(b for b in page["blocks"] if NESTED_MARK in (b.get("text") or ""))
    prefix, _ = reconstruct_split(sw["text"])
    print(f"{cid} p{pn}: oryginał len={len(sw['text'])}, prefix po rekonstrukcji len={len(prefix)}")
    print(f"  prefix tail: {prefix[-120:]!r}")

III p44: 3 bloków, w tym 1 połykacz(y) — pozostałe 2 bez zmian
VII p137: 11 bloków, w tym 1 połykacz(y) — pozostałe 10 bez zmian
X p179: 3 bloków, w tym 1 połykacz(y) — pozostałe 2 bez zmian

III p44: oryginał len=2841, prefix po rekonstrukcji len=160
  prefix tail: 'ansowe w BGK, a wieczorem zmienia się w DJ-a, serwując chilloutowe brzmienia w swojej audycji „blokz" w Radiu Kapitał."}'
VII p137: oryginał len=1555, prefix po rekonstrukcji len=235
  prefix tail: ' w życie Rozporządzenia CRR3,\n- dostosowanie wymogów w zakresie bezpieczeństwa cyfrowego zgodnie z Rozporządzeniem „DORA'
X p179: oryginał len=4931, prefix po rekonstrukcji len=192
  prefix tail: 'o przeprowadzono zgodnie ze standardem „The Greenhouse Gas Protocol (GHG Protocol)" obejmującym następujące dokumenty:"}'


In [ ]:
# Ryzyko 3: czy brakujące section-header z ToC są w pełni pokryte przez rekonstrukcję?
for cid, pn in [("III", 44), ("VII", 137), ("X", 179)]:
    ch = json.loads((CACHE / f"{cid}.json").read_text(encoding="utf-8"))
    page = next(p for p in ch["pages"] if p["page_num"] == pn)
    sw = next(b for b in page["blocks"] if NESTED_MARK in (b.get("text") or ""))
    _, blocks_a = reconstruct_split(sw["text"])
    recovered_sh = {normalize(tx) for et, tx in blocks_a if et == "section-header"}
    missing_before = {
        normalize(s) for s in page["sections"]
        if normalize(s) not in {normalize(b["text"]) for b in page["blocks"] if b["element_type"] == "section-header"}
    }
    recovered_from_missing = missing_before & recovered_sh
    still_missing = missing_before - recovered_sh
    print(f"{cid} p{pn}: brakowało {len(missing_before)} section-header z ToC; rekonstrukcja odzyskuje {len(recovered_from_missing)}; nadal brak: {still_missing}")

III p44: brakowało 0 section-header z ToC; rekonstrukcja odzyskuje 0; nadal brak: set()
VII p137: brakowało 1 section-header z ToC; rekonstrukcja odzyskuje 1; nadal brak: set()
X p179: brakowało 0 section-header z ToC; rekonstrukcja odzyskuje 0; nadal brak: set()


## Sekcja 5 — Rekomendacja

Wypełniamy na podstawie wyników powyżej. Oczekiwane kryteria decyzji:

| Kryterium | Jeśli spełnione | Decyzja |
|---|---|---|
| Rekonstrukcja A odzyskuje **wszystkie** brakujące section-header z ToC | tak | naprawa skryptem z `text` jest wystarczająca — retry Vision zbędny |
| Prefix (prawdziwa treść bloku-połykacza) wygląda na kompletny/niezaburzony | tak | jw. |
| Pozostaje jakiś section-header z ToC niedostępny | tak | retry Vision dla tej jednej strony |
| Liczba połykaczy na stronie > 1 | nieprawdopodobne (widzimy po 1) | retry Vision |

Osobna decyzja — **pre-processor w produkcji** — rozważamy w kolejnym kroku. Pre-processor zapobiega NOWYM połknięciom przy przyszłych ekstrakcjach; nie naprawia istniejącego cache.